# Multilingual RAG — German, French and Italian (FDV)

Indexes **all three language versions** (`data/de/`, `data/fr/`, `data/it/`).  

### Content
* Section 1: Imports and configuration
* Section 2: Document loading and chunking
* Section 3: Models
* Section 4: Indexing
* Section 5: BM25 Index
* Section 6: Retrieval test
* Section 7: Single query answer generation test

## Section 1 — Imports & Configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

from chunking    import load_documents, build_chunk_records, chunking_fingerprint
from retrieval   import build_bm25_index, retrieve
from generation  import ask
from config      import EMBED_MODEL, CE_MODEL, BATCH_SIZE, TOP_K, BM25_WEIGHT, SEMANTIC_POOL, RERANK_POOL, GENERATION_MODEL

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIRS       = {
    "de": Path("data/de"),
    "fr": Path("data/fr"),
    "it": Path("data/it"),
}
COLLECTION_NAME = "fdv_multilingual"
CHROMA_DIR      = Path("chroma_db_multilingual")
MODEL_NAME      = GENERATION_MODEL

CFG = {
    "TOP_K":          TOP_K,
    "BM25_WEIGHT":    BM25_WEIGHT,
    "SEMANTIC_POOL":  SEMANTIC_POOL,   # TOP_K * 10
    "RERANK_POOL":    RERANK_POOL,    # TOP_K * 2
    "MODEL_NAME":     MODEL_NAME,
}

print("Configuration ready.")

c:\Users\niw\Documents\CAS_NLP_Project\Project_tRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration ready.


## Section 2 — Document Loading & Chunking

In [2]:
all_docs = []
for lang, data_dir in DATA_DIRS.items():
    docs = load_documents(data_dir, lang)
    all_docs.extend(docs)
    print(f"  {lang}: {len(docs)} documents loaded")

chunk_records = build_chunk_records(all_docs)
print(f"\nTotal chunks: {len(chunk_records)}")

# Chunk count per language
lang_counts = {}
for r in chunk_records:
    lang_counts[r["language"]] = lang_counts.get(r["language"], 0) + 1
for lang, count in sorted(lang_counts.items()):
    print(f"  {lang}: {count} chunks")

# Preview
preview = pd.DataFrame([
    {k: v for k, v in r.items() if k != "text"}
    for r in chunk_records[:5]
])
display(preview)

  de: 15 documents loaded
  fr: 15 documents loaded
  it: 15 documents loaded

Total chunks: 4272
  de: 1360 chunks
  fr: 1482 chunks
  it: 1430 chunks


,source_file,document_title,regulation_number,section_id,section_title,chunk_index,language
0,01_Grundlagen.txt,Grundlagen,R 300.1,NaN,NaN,0,de
1,01_Grundlagen.txt,Grundlagen,R 300.1,1.2,Geltungsbereich,1,de
2,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.1,Anwendbarkeit der Vorgaben nach Teil-Geltungsb...,2,de
3,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.2,Anwendbarkeit der Vorgaben nach Funktionen,3,de
4,01_Grundlagen.txt,Grundlagen,R 300.1,1.2.3,Auswirkungen des europäischen Rechts,4,de


## Section 3 — Models
Load the two models needed for retrieval: `intfloat/multilingual-e5-small` (converts text into vectors) and `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (reranker) and run a small cross-lingual similarity demo to validate.

In [5]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading cross-encoder...")
cross_encoder = CrossEncoder(CE_MODEL)

print("\nModel demo — cross-lingual similarity:")
demo_passages = [
    "passage: Bremsprobe im Störungsfall",
    "passage: Zugvorbereitung und Zugbildung",
    "passage: Essai de frein en cas de dérangement",
]
demo_query = "query: Comment effectuer la vérification du frein à main?"

demo_embs = embedder.encode(demo_passages + [demo_query], normalize_embeddings=True)
q_emb     = demo_embs[-1]
sims      = demo_embs[:-1] @ q_emb

print(f"\nDemo query: {demo_query}")
for passage, sim in zip(demo_passages, sims):
    print(f"  sim={sim:.3f}  {passage[:60]}")

Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3319.24it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading cross-encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2354.81it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model demo — cross-lingual similarity:

Demo query: query: Comment effectuer la vérification du frein à main?
  sim=0.806  passage: Bremsprobe im Störungsfall
  sim=0.771  passage: Zugvorbereitung und Zugbildung
  sim=0.841  passage: Essai de frein en cas de dérangement


## Section 4 — Indexing

In [6]:
fp         = chunking_fingerprint()
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine",
              "chunk_min":     fp["chunk_min"],
              "chunk_max":     fp["chunk_max"],
              "chunk_overlap": fp["chunk_overlap"]},
)

if collection.count() > 0:
    print(f"Collection already contains {collection.count()} chunks — skipping re-indexing.")
else:
    print(f"Embedding and indexing {len(chunk_records)} chunks...")
    texts_to_embed = ["passage: " + r["text"] for r in chunk_records]

    all_embeddings = []
    for i in tqdm(range(0, len(texts_to_embed), BATCH_SIZE), desc="Embedding"):
        batch = texts_to_embed[i : i + BATCH_SIZE]
        embs  = embedder.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        all_embeddings.extend(embs.tolist())

    CHROMA_BATCH = 5000
    for i in tqdm(range(0, len(chunk_records), CHROMA_BATCH), desc="Indexing"):
        batch_records = chunk_records[i : i + CHROMA_BATCH]
        batch_embs    = all_embeddings[i : i + CHROMA_BATCH]
        metadatas = [
            {k: (v if v is not None else "") for k, v in r.items() if k != "text"}
            for r in batch_records
        ]
        collection.add(
            ids=[str(i + j) for j in range(len(batch_records))],
            embeddings=batch_embs,
            documents=[r["text"] for r in batch_records],
            metadatas=metadatas,
        )

    print(f"Indexed {collection.count()} chunks.")

assert collection.count() == len(chunk_records), (
    f"ChromaDB has {collection.count()} chunks but chunk_records has "
    f"{len(chunk_records)}. Delete the chroma_db_* folder and re-run the indexing cell."
)

Embedding and indexing 4272 chunks...


Indexing: 100%|██████████| 1/1 [00:06<00:00,  6.26s/it]

Indexed 4272 chunks.


## Section 5 — BM25 Index

In [7]:
print("Building BM25 index...")
bm25_index = build_bm25_index(chunk_records)
print(f"BM25 index ready ({len(chunk_records)} documents).")

Building BM25 index...
BM25 index ready (4272 documents).


## Section 6 — Retrieval Test

Edit `TEST_QUERIES` to try different questions.

In [8]:
TEST_QUERIES = [
    "Wie ist vorzugehen, wenn eine Zahnstangenweiche aufgeschnitten wurde?",
    "Comment faut-il procéder si une aiguille de voie à crémaillère a été talonnée?",
    "Come si procede se è stato tallonato uno scambio a cremagliera?",
]

In [9]:
for query in TEST_QUERIES:
    print(f"\n{'─'*70}")
    print(f"Query: {query}")
    print(f"{'─'*70}")
    results = retrieve(query, embedder, cross_encoder,
                       collection, bm25_index, chunk_records, CFG)
    for j, r in enumerate(results, 1):
        sec  = r.get("section_id") or ""
        lang = r.get("language")   or ""
        print(f"  [{j}] [{lang.upper()}] {r.get('regulation_number','')} section {sec}  "
              f"rerank={r['rerank_score']:.3f}  "
              f"sem={r['sem_score']:.3f}  "
              f"bm25={r['bm25_score']:.3f}")
        print(f"       {r['text'][:120].strip()}...")


──────────────────────────────────────────────────────────────────────
Query: Wie ist vorzugehen, wenn eine Zahnstangenweiche aufgeschnitten wurde?
──────────────────────────────────────────────────────────────────────
  [1] [DE] R 300.9 section 4.6  rerank=5.012  sem=0.914  bm25=1.000
       4.6	Weichenaufschneidung
4.6.1	Grundsatz
	
	Das Aufschneiden von Weichen ist verboten, da es betriebsgefährdende Beschäd...
  [2] [DE] R 300.9 section 4.6.3  rerank=0.997  sem=0.846  bm25=0.272
       4.6.3	Kontrolle einer aufgeschnittenen Weiche
	
	Die Kontrolle einer aufgeschnittenen Weiche hat grundsätzlich durch den...

──────────────────────────────────────────────────────────────────────
Query: Comment faut-il procéder si une aiguille de voie à crémaillère a été talonnée?
──────────────────────────────────────────────────────────────────────
  [1] [FR] R 300.9 section 4.6.2  rerank=10.814  sem=0.932  bm25=1.000
       4.6.2	Aiguille talonnée
	
	Lorsqu’une aiguille a été talonnée, il faut ar

## Section 7 — Single Query Answer

Edit `QUERY` to ask a different question.

In [11]:
QUERY = "Wann muss der Lokführer die Wirkung der Luftbremse prüfen?"

In [12]:
print(f"Query: {QUERY}\n")
answer, chunks = ask(QUERY, embedder, cross_encoder,
                     collection, bm25_index, chunk_records, CFG)

print("Answer:")
print(answer)

print("\nSources used:")
for r in chunks:
    sec  = r.get("section_id")    or ""
    ttl  = r.get("section_title") or ""
    lang = r.get("language")      or ""
    print(f"  [{lang.upper()}] {r.get('regulation_number','')} §{sec} — {ttl}")

Query: Wann muss der Lokführer die Wirkung der Luftbremse prüfen?

Answer:
Basierend auf den bereitgestellten Auszügen muss der Lokführer die Wirkung der Luftbremse in folgenden Fällen prüfen:

*   Unmittelbar nach der Abfahrt (2.3.7)
*   Nach Veränderungen an der Zugzusammensetzung (2.3.7)
*   Nach einem LF-Wechsel (2.3.7)
*   Vor der Einfahrt in starkes Gefälle oder Kopfbahnhöfe (2.3.7)
*   Zeitweise nach der letzten Bremsung mit der automatischen Bremse bei Flugschnee oder großer Kälte (2.3.7)

Sources used:
  [DE] R 300.14 §2.3.7 — Bremsprobe auf Wirkung bei Zügen
  [DE] R 300.14 §2.3.5 — Bremsprobe bei Triebfahrzeugen
